In [ ]:
import pandas as pd
import numpy as np
import torch
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import MinMaxScaler
import copy
import pickle

class ForwardNet(nn.Module):
  def __init__(self, in_dim=6, out_dim=3):
    super().__init__()
    self.net = nn.Sequential(
        nn.Linear(in_dim, 512),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(512, 256),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(256, 128),
        nn.ReLU(),
        nn.Dropout(0.2),
        nn.Linear(128, out_dim)
    )

  def forward(self, x):
    return self.net(x)

def train_forward_net(forward_net, train_loader, val_loader, epochs = 100, device = 'cuda', lr = 1e-3):
  forward_net.to(device)
  opt = optim.AdamW(forward_net.parameters(), lr=lr)
  train_losses = []
  val_losses = []
  best_val_loss = float("inf")
  best_state = None

  for epoch in range(epochs):
    forward_net.train()
    total_train = 0.0
    for y, x in train_loader:
      y = y.to(device)
      x = x.to(device)
      opt.zero_grad()
      y_hat = forward_net(x)
      loss = F.mse_loss(y_hat, y, reduction="mean")
      loss.backward()
      opt.step()
      total_train += loss.item()*x.size(0)
    train_epoch_loss = total_train / len(train_loader.dataset)
    train_losses.append(train_epoch_loss)
    forward_net.eval()
    total_val = 0.0
    for y, x in val_loader:
      y = y.to(device)
      x = x.to(device)

      y_hat = forward_net(x)
      val_loss = F.mse_loss(y_hat, y, reduction="mean")
      total_val += val_loss.item() * x.size(0)

    val_epoch_loss = total_val / len(val_loader.dataset)
    val_losses.append(val_epoch_loss)

    if val_epoch_loss < best_val_loss:
      best_val_loss = val_epoch_loss
      best_state = copy.deepcopy(forward_net.state_dict())

    print(f"[Forward Net] Epoch: {epoch+1}, Train loss: {train_epoch_loss:.4f} Val loss: {val_epoch_loss:.4f}")
    if best_state is not None:
      forward_net.load_state_dict(best_state)
      print(f"Loaded best model with Val Loss: {best_val_loss:.4f}")
  return forward_net

class CVAE(nn.Module):
  def __init__(self):
    super(CVAE, self).__init__()

    self.fc1 = nn.Linear(9, 512)  # 7 columns + 2 conditions
    self.fc2 = nn.Linear(512, 256)
    self.fc3 = nn.Linear(256, 128)
    self.fc21 = nn.Linear(128, 30)
    self.fc22 = nn.Linear(128, 30)
    self.dropout = nn.Dropout(p=0.2)

    self.fc4 = nn.Linear(33, 128)  # 30 latent space + 2 conditions
    self.fc5 = nn.Linear(128, 256)
    self.fc6 = nn.Linear(256, 512)
    self.fc7 = nn.Linear(512, 6)

  def encode(self, x, condition):
    combined = torch.cat([x, condition], 1)
    h1 = F.relu(self.fc1(combined))
    h2 = F.relu(self.fc2(self.dropout(h1)))
    h3 = F.relu(self.fc3(self.dropout(h2)))
    return self.fc21(h3), self.fc22(h3)

  def reparameterize(self, mu, logvar):
    std = torch.exp(0.5*logvar)
    eps = torch.randn_like(std)
    return mu + eps*std

  def decode(self, z, condition):
    combined = torch.cat([z, condition], 1)
    h4 = F.relu(self.fc4(combined))
    h5 = F.relu(self.fc5(self.dropout(h4)))
    h6 = F.relu(self.fc6(self.dropout(h5)))
    return self.fc7(self.dropout(h6))

  def forward(self, x, condition):
    mu, logvar = self.encode(x, condition)
    z = self.reparameterize(mu, logvar)
    return self.decode(z, condition), mu, logvar

def cvae_loss(recon_x, x, mu, logvar, forward_net, y, beta=0.001, lam_phys=5.0):
  MSE = F.mse_loss(recon_x, x, reduction='mean')
  KLD = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
  base = MSE + beta*KLD

  y_hat = forward_net(recon_x)
  phys = F.mse_loss(y_hat, y, reduction='mean')

  return base  + lam_phys*phys, base.item(), phys.item()

def train_cvae(cvae, forward_net, train_loader, val_loader, epochs=300, patience=50, beta=0.001, lam_phys=5.0, lr=1e-3, device="cuda"):

  cvae.to(device)
  forward_net.to(device)

  forward_net.eval()

  for p in forward_net.parameters():
    p.requires_grad = False

  optimizer = optim.AdamW(cvae.parameters(), lr=lr)
  scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=30, gamma=0.7)

  best_val_loss = float("inf")
  best_state = None
  no_improve_epochs = 0

  train_losses = []
  val_losses = []


  for epoch in range(epochs):
    cvae.train()
    train_total = 0.0
    train_base_total = 0.0
    train_phys_total = 0.0

    for y, x in train_loader:
      y = y.to(device)
      x = x.to(device)

      optimizer.zero_grad()

      recon_batch, mu, logvar = cvae(x, y)
      loss, base_loss, phys_loss = cvae_loss(
                recon_batch, x, mu, logvar,
                forward_net, y,
                beta=beta,
                lam_phys=lam_phys
            )

      loss.backward()
      optimizer.step()

      train_total += loss.item() * x.size(0)
      train_base_total += base_loss * x.size(0)
      train_phys_total += phys_loss *x.size(0)

    avg_train_loss = train_total / len(train_loader.dataset)
    avg_train_base = train_base_total / len(train_loader.dataset)
    avg_train_phys = train_phys_total / len(train_loader.dataset)
    train_losses.append(avg_train_loss)

    cvae.eval()
    val_total = 0.0
    val_base_total = 0.0
    val_phys_total = 0.0

    with torch.no_grad():
      for y, x in val_loader:
        y = y.to(device)
        x = x.to(device)

        recon_batch, mu, logvar = cvae(x, y)
        val_loss, val_base_loss, val_phys_loss = cvae_loss(
                        recon_batch, x, mu, logvar,
                        forward_net, y,
                        beta=beta,
                        lam_phys=lam_phys
                    )


        val_total += val_loss.item() * x.size(0)
        val_base_total += val_base_loss * x.size(0)
        val_phys_total += val_phys_loss * x.size(0)

    avg_val_loss = val_total / len(val_loader.dataset)
    avg_val_base = val_base_total / len(val_loader.dataset)
    avg_val_phys = val_phys_total / len(val_loader.dataset)
    val_losses.append(avg_val_loss)

    print(
                f"[PINN-CVAE] Epoch {epoch+1}, "
                f"Train Total: {avg_train_loss:.4f}, Train Base: {avg_train_base:.4f}, Train Phys: {avg_train_phys:.4f}, "
                f"Val Total: {avg_val_loss:.4f}, Val Base: {avg_val_base:.4f}, Val Phys: {avg_val_phys:.4f}"
            )

    if avg_val_loss < best_val_loss:
      best_val_loss = avg_val_loss
      best_state = copy.deepcopy(cvae.state_dict())
      no_improve_epochs = 0
    else:
      no_improve_epochs += 1
      if no_improve_epochs >= patience:
        print(
                        f"Early stopping at epoch {epoch+1}. "
                        f"No improvement in validation loss for {patience} consecutive epochs."
                    )
        break
    scheduler.step()

  if best_state is not None:
    cvae.load_state_dict(best_state)
    print(f"Loaded best CVAE model with Val Loss: {best_val_loss:.4f}")
  return cvae

def generate_outputs(model, input_data, num_samples=10):
  model.eval()
  device = next(model.parameters()).device  # get model device

  with torch.no_grad():
    input_df = pd.DataFrame(
    input_data,columns=['Frequency (Hz)', 'Storage modulus (Pa)', 'Loss modulus (Pa)'])
    input_data_scaled = Y_scaler.transform(input_df)

    conditions = torch.tensor(input_data_scaled, dtype=torch.float32, device=device)

    outputs = []
    for _ in range(num_samples):
      z = torch.randn(conditions.size(0), 30, device=device)
      output = model.decode(z, conditions)
      output = X_scaler.inverse_transform(output.cpu().numpy())
      outputs.append(output)

  return outputs

In [ ]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.set_default_dtype(torch.float32)


device = 'cuda' if torch.cuda.is_available() else "cpu"

df_train = pd.read_csv('/content/train.csv')
df_val = pd.read_csv('/content/val.csv')


X_train, X_val = df_train.iloc[:, 0:6], df_val.iloc[:, 0:6]
y_train, y_val = df_train.iloc[:, 6:9], df_val.iloc[:, 6:9]

with open('/content/X_scaler_cvae.pkl', 'rb') as f:
  X_scaler = pickle.load(f)

with open('/content/Y_scaler_cvae.pkl', 'rb') as f:
  Y_scaler = pickle.load(f)


X_train_scaled = X_scaler.transform(X_train)
y_train_scaled = Y_scaler.transform(y_train)

X_val_scaled = X_scaler.transform(X_val)
y_val_scaled = Y_scaler.transform(y_val)

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_scaled, dtype=torch.float32)

train_data = TensorDataset(y_train_tensor, X_train_tensor)
train_loader = DataLoader(train_data, batch_size=32, shuffle=True)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_scaled, dtype=torch.float32)

val_data = TensorDataset(y_val_tensor, X_val_tensor)
val_loader = DataLoader(val_data, batch_size=32, shuffle=False)


cvae = CVAE()
forward_net = ForwardNet(in_dim=6, out_dim=3)
best_forward = train_forward_net(forward_net, train_loader, val_loader, epochs = 100, device = device)
best_cvae = train_cvae(cvae, best_forward, train_loader, val_loader, epochs = 300, patience= 20, beta=0.001, lam_phys = 2, device=device)


df_test = pd.read_csv("/content/test.csv")

frequency, storage_modulus, loss_modulus = df_test['Frequency (Hz)'], df_test['Storage modulus (Pa)'], df_test['Loss modulus (Pa)']


# Initialize an empty DataFrame with specified columns
columns = ['Acrylamide Conc. %', 'Bis-acrylamide conc %', 'Photo-initiator conc. %', 'Layer Height. (micron)',
          'Bottom Layer exposure time (s) ', 'Exposure time (s)', 'Frequency (Hz)',
          'Storage modulus (Pa)', 'Loss modulus (Pa)']

rows = []

for i, j, k in zip(frequency, storage_modulus, loss_modulus):
    input_data = [[i, j, k]]
    output_parameters = generate_outputs(best_cvae, input_data, num_samples=1)[0]
    combined_data = list(output_parameters[0]) + [i, j, k]
    rows.append(combined_data)

df = pd.DataFrame(rows, columns=columns)

df.to_csv('synth_cvae_pinn.csv', index=False)
print("\nSaved synthetic data!")

[Forward Net] Epoch: 1, Train loss: 0.6053 Val loss: 0.5586
Loaded best model with Val Loss: 0.5586
[Forward Net] Epoch: 2, Train loss: 0.5205 Val loss: 0.5569
Loaded best model with Val Loss: 0.5569
[Forward Net] Epoch: 3, Train loss: 0.5176 Val loss: 0.5336
Loaded best model with Val Loss: 0.5336
[Forward Net] Epoch: 4, Train loss: 0.5115 Val loss: 0.5301
Loaded best model with Val Loss: 0.5301
[Forward Net] Epoch: 5, Train loss: 0.5075 Val loss: 0.5208
Loaded best model with Val Loss: 0.5208
[Forward Net] Epoch: 6, Train loss: 0.5094 Val loss: 0.5177
Loaded best model with Val Loss: 0.5177
[Forward Net] Epoch: 7, Train loss: 0.5073 Val loss: 0.5184
Loaded best model with Val Loss: 0.5177
[Forward Net] Epoch: 8, Train loss: 0.5098 Val loss: 0.5266
Loaded best model with Val Loss: 0.5177
[Forward Net] Epoch: 9, Train loss: 0.5140 Val loss: 0.5305
Loaded best model with Val Loss: 0.5177
[Forward Net] Epoch: 10, Train loss: 0.5105 Val loss: 0.5323
Loaded best model with Val Loss: 0.5177